# Notebook — Génération (Phases 1-3)

Outil TAF → X-Plane 12 → IoU. Équivaut à `run_pipeline.py generate / export / evaluate`.
Les fonctionnalités annexes (dataset, regroup, sanity, vidéo…) sont dans `notebook/features.ipynb`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "run_pipeline.py").exists():
            return p
    return start

ROOT = _find_root(Path.cwd())

_project = str(ROOT / "project")
if _project not in sys.path:
    sys.path.insert(0, _project)

print(f"ROOT = {ROOT}")

## 2. Generate (Phase 1 — TAF seul, offline)

Équivalent CLI : `py run_pipeline.py generate -n N`

Sample N scénarios via TAF/z3 et écrit `runs/<generation>/<ICAO_RWY>/` avec `.yaml`,
`poses_cam_export.json`, `fault_profile.json`, `weather_profile.json`.
Pas besoin de X-Plane. 
Si `nb_scenarios=None` => valeur de `project/settings.xml`.

In [ ]:
from Generate import generate_runs

# generate_runs(nb_scenarios=5)
generate_runs(nb_scenarios=2);

## 3. Export (Phase 2 — Rendu X-Plane + fautes capteur)

Équivalent CLI :
- `py run_pipeline.py export generation_01/LFPO_25`
- `py run_pipeline.py export --all --generation generation_01`

Supposé que la Phase 1 (`generate`) a déjà été faite. Nécessite X-Plane 12
lancé (mode fenêtre, scaling 100%). Sortie : `footage/`, `degraded/`.


In [ ]:
import xml.etree.ElementTree as ET
from runs import render_runs

# Répertoire X-Plane lu depuis project/settings.xml -> Chemin du simulateur
_settings = {p.attrib["name"]: p.attrib["value"]
             for p in ET.parse(ROOT / "project" / "settings.xml").getroot()}
XPLANE_DIR = _settings["xplane_dir"]

# Un seul run
# render_runs(run_name="generation_01/KLAX_25R", xplane_dir=XPLANE_DIR)

# Tous les runs :
render_runs(all_runs=True, xplane_dir=XPLANE_DIR);

## 4. Evaluate (Phase 3 — GT LARD + YOLO + IoU, sans X-Plane)

Équivalent CLI : `py run_pipeline.py evaluate --all`.

Suppose que `footage/` ou `degraded/` existe déjà (Phase 2 faite). YOLO predict + IoU. Ré-exécutable à volonté
avec d'autres poids/seuils sans re-rendre.

In [ ]:
from runs import evaluate_runs

# Un seul run (nom seul si unique, sinon chemin composé '<gen>/<run>') :
# evaluate_runs(run_name="generation_01/KLAX_25R")

# Ré-évaluation avec seuils personnalisés (conf relance YOLO, pas de re-rendu) :
# evaluate_runs(
#     all_runs=True,
#     conf=0.25,          # seuil de confiance YOLO (filtre les détections)
#     imgsz=512,          # taille d'inférence YOLO
#     iou_thresh=0.5,     # seuil IoU
#     iou_method="CIOU",  # IOU / GIOU / DIOU / CIOU
#     runway=None,        # filtrer la GT sur une piste (ex: "24"), None = toutes
# )

# Tous les runs :
evaluate_runs(all_runs=True);